# Generalized Powerline Folder Visualization (Real-Only)

Uses only `*_real.bin` files — no `.img` files.

Visualizations:
- signal statistics
- time-domain plots
- spectrograms (**10 kHz bands, chirp vs test**)
- PSD comparison
- audio preview

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram, welch
from IPython.display import Audio, display

# =========================
# CONFIGURATION
# =========================
PROJECT_ROOT = '<REPO_ROOT>'
TARGET_FOLDER = os.path.join(PROJECT_ROOT, 'sound_bar')  # change to any folder
SAMPLE_RATE = 200_000

TIME_WINDOW_SEC = 30
SPEC_WINDOW_SEC = 30
N_PER_SEG_SPEC = 2048
N_PER_SEG_PSD = 16384

print('TARGET_FOLDER:', TARGET_FOLDER)
print('SAMPLE_RATE  :', SAMPLE_RATE)

In [ ]:
# Load chirp and test real signals
signals = {}
for name in ('chirp', 'test'):
    path = os.path.join(TARGET_FOLDER, f'{name}_real.bin')
    if os.path.exists(path):
        signals[name] = np.fromfile(path, dtype=np.float32)
        print(f'Loaded {name}: {len(signals[name]):,} samples  ({len(signals[name]) / SAMPLE_RATE:.2f}s)')
    else:
        print(f'WARNING: {path} not found')

if not signals:
    raise RuntimeError('No chirp_real.bin or test_real.bin found in TARGET_FOLDER')

In [ ]:
for name, sig in signals.items():
    print(f'\n--- {name} ---')
    print(f'  Samples        : {len(sig):,}')
    print(f'  Duration (sec) : {len(sig) / SAMPLE_RATE:.2f}')
    print(f'  Min/Max        : [{sig.min():.6f}, {sig.max():.6f}]')
    print(f'  Mean/Std       : {sig.mean():.6f} / {sig.std():.6f}')
    print(f'  RMS            : {np.sqrt(np.mean(sig**2)):.6f}')

## Time Domain (First N Seconds)

In [ ]:
colors = {'chirp': 'orange', 'test': 'steelblue'}
n = len(signals)
fig, axes = plt.subplots(n, 1, figsize=(14, 4 * n), sharex=True, squeeze=False)

for ax, (name, sig) in zip(axes[:, 0], signals.items()):
    plot_samples = min(int(TIME_WINDOW_SEC * SAMPLE_RATE), len(sig))
    t = np.arange(plot_samples) / SAMPLE_RATE
    ax.plot(t, sig[:plot_samples], color=colors.get(name, 'gray'), linewidth=0.5)
    ax.set_title(f"{name} — Real Signal (First {plot_samples / SAMPLE_RATE:.1f}s)")
    ax.set_ylabel('Amplitude')
    ax.grid(True, alpha=0.3)

axes[-1, 0].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## Spectrograms — chirp vs test (10 kHz bands, 0 to Nyquist)

In [ ]:
# Compute spectrograms for each signal
nyquist = SAMPLE_RATE // 2
band_hz = 10_000
bands = [(blo, blo + band_hz) for blo in range(0, nyquist, band_hz)]
n_bands = len(bands)
file_names = list(signals.keys())
n_files = len(file_names)

specs, freqs, times = {}, {}, {}
for name, sig in signals.items():
    spec_samples = min(int(SPEC_WINDOW_SEC * SAMPLE_RATE), len(sig))
    f, t_s, S = spectrogram(sig[:spec_samples], fs=SAMPLE_RATE, nperseg=N_PER_SEG_SPEC)
    specs[name] = 10 * np.log10(S + 1e-12)
    freqs[name] = f
    times[name] = t_s

# rows = 10 kHz bands, cols = chirp | test
fig, axes = plt.subplots(n_bands, n_files, figsize=(9 * n_files, n_bands * 3), squeeze=False)

for row, (flo, fhi) in enumerate(bands):
    for col, name in enumerate(file_names):
        ax = axes[row][col]
        f = freqs[name]
        mask = (f >= flo) & (f <= fhi)
        if not mask.any():
            ax.set_visible(False)
            continue
        S_band = specs[name][mask, :]
        f_band = f[mask]
        t_s = times[name]
        vmin = np.percentile(S_band, 5)
        vmax = np.percentile(S_band, 95)
        im = ax.imshow(
            S_band,
            origin='lower',
            aspect='auto',
            extent=[t_s.min(), t_s.max(), f_band.min() / 1000, f_band.max() / 1000],
            cmap='viridis',
            vmin=vmin,
            vmax=vmax,
        )
        ax.set_title(f"{name}  {flo // 1000}–{fhi // 1000} kHz")
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Freq (kHz)')
        plt.colorbar(im, ax=ax, label='dB')

fig.suptitle('Spectrogram by 10 kHz Bands — chirp vs test', fontsize=15)
plt.tight_layout()
plt.show()

## PSD Comparison

In [ ]:
colors = {'chirp': 'orange', 'test': 'steelblue'}
n = len(signals)
fig, axes = plt.subplots(n, 1, figsize=(13, 5 * n), sharex=True, squeeze=False)

for ax, (name, sig) in zip(axes[:, 0], signals.items()):
    nperseg = min(N_PER_SEG_PSD, len(sig))
    f_psd, P_psd = welch(sig, fs=SAMPLE_RATE, nperseg=nperseg)
    P_db = 10 * np.log10(P_psd + 1e-20)
    ax.plot(f_psd, P_db, color=colors.get(name, 'gray'))
    ax.set_title(f"{name} — PSD")
    ax.set_ylabel('Power (dB/Hz)')
    ax.grid(True, alpha=0.3)

axes[-1, 0].set_xlabel('Frequency (Hz)')
plt.xlim(0, SAMPLE_RATE / 2)
plt.tight_layout()
plt.show()

## Audio Preview

In [ ]:
for name, sig in signals.items():
    dur = len(sig) / SAMPLE_RATE
    preview_sec = min(30, int(dur))
    preview_samples = int(preview_sec * SAMPLE_RATE)
    audio_sig = sig[:preview_samples]
    audio_sig = audio_sig / (np.max(np.abs(audio_sig)) + 1e-12)
    print(f'Playing {name}: {preview_sec}s @ {SAMPLE_RATE} Hz')
    display(Audio(audio_sig, rate=SAMPLE_RATE))

## Residual Analysis (Harmonic-Subtracted) — test signal

In [ ]:
from scipy.signal import butter, sosfilt, spectrogram as sp_spectrogram

sig_test = signals['test'].astype(np.float64)
N = len(sig_test)

# ── 1. Find exact fundamental and subtract all harmonics via FFT comb ──
F = np.fft.rfft(sig_test)
bin_freqs = np.fft.rfftfreq(N, d=1/SAMPLE_RATE)
idx_lo, idx_hi = np.searchsorted(bin_freqs, 55), np.searchsorted(bin_freqs, 65)
f0 = bin_freqs[idx_lo + np.argmax(np.abs(F[idx_lo:idx_hi]))]
print(f'Fundamental: {f0:.4f} Hz')

bin_width = SAMPLE_RATE / N
F_clean = F.copy()
n_harm = 0
h = 1
while h * f0 < SAMPLE_RATE / 2:
    cb = int(round(h * f0 / bin_width))
    F_clean[max(0, cb - 4):min(len(F_clean), cb + 5)] = 0
    h += 1
    n_harm += 1
print(f'Removed {n_harm} harmonics')

residual = np.fft.irfft(F_clean, n=N)

# ── 2. Time-domain plot of residual (first 30s) ──
plot_samples = min(int(TIME_WINDOW_SEC * SAMPLE_RATE), N)
t = np.arange(plot_samples) / SAMPLE_RATE

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t, residual[:plot_samples], color='steelblue', linewidth=0.4)
ax.set_title('test — Residual (after removing all 60 Hz harmonics)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── 3. Zoomed time-domain: speech band only (300–3400 Hz) ──
sos_speech = butter(4, [300/(SAMPLE_RATE/2), 3400/(SAMPLE_RATE/2)], btype='band', output='sos')
res_speech = sosfilt(sos_speech, residual)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t, res_speech[:plot_samples], color='darkorange', linewidth=0.5)
ax.set_title('test — Residual (bandpassed 300–3400 Hz speech band)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── 4. Spectrogram of residual (0–8 kHz) ──
spec_samples = min(int(SPEC_WINDOW_SEC * SAMPLE_RATE), N)
f_s, t_s, S = sp_spectrogram(residual[:spec_samples], fs=SAMPLE_RATE, nperseg=N_PER_SEG_SPEC)
S_db = 10 * np.log10(S + 1e-12)
mask = f_s <= 8000

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(
    S_db[mask],
    origin='lower', aspect='auto',
    extent=[t_s.min(), t_s.max(), f_s[mask].min(), f_s[mask].max()],
    cmap='inferno',
    vmin=np.percentile(S_db[mask], 5),
    vmax=np.percentile(S_db[mask], 98),
)
ax.set_title('test — Residual Spectrogram (0–8 kHz)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
plt.colorbar(im, ax=ax, label='Power (dB)')
plt.tight_layout()
plt.show()

# ── 5. PSD: original vs residual, zoomed to speech band ──
from scipy.signal import welch as sc_welch
f_o, P_o = sc_welch(sig_test,  fs=SAMPLE_RATE, nperseg=65536)
f_r, P_r = sc_welch(residual,  fs=SAMPLE_RATE, nperseg=65536)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(f_o, 10*np.log10(P_o+1e-30), color='gray',       alpha=0.6, linewidth=0.8, label='Original')
ax.plot(f_r, 10*np.log10(P_r+1e-30), color='darkorange',  linewidth=0.8, label='Residual')
ax.set_xlim(0, 8000)
ax.set_title('test — PSD: Original vs Residual (speech band zoom)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (dB/Hz)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── 6. Frame-level RMS energy of speech-band residual ──
frame_len = int(0.02 * SAMPLE_RATE)  # 20 ms
n_frames = len(res_speech) // frame_len
frame_rms = np.array([
    np.sqrt(np.mean(res_speech[i*frame_len:(i+1)*frame_len]**2))
    for i in range(n_frames)
])
frame_t = np.arange(n_frames) * 0.02

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(frame_t, frame_rms, color='crimson', linewidth=0.8)
ax.set_title('test — Speech-band Residual Energy (20ms frames) — bursty = speech-like')
ax.set_xlabel('Time (s)')
ax.set_ylabel('RMS')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nCoV (std/mean): {frame_rms.std()/frame_rms.mean():.3f}  (>0.3 = speech-like)')
print(f'Max/Min ratio : {frame_rms.max()/frame_rms.min():.2f}x  (>5x = speech-like)')